Formataçãodo do dados do DataFrame de Infodengue 

In [1]:
import pandas as pd
import requests
import io
import time

# Base escolhida: InfoDengue
# Cidade analisada: Fortaleza
# Período da consulta: 2017 até 2024

geocode = 2304400
doencas = ["dengue", "zika", "chikungunya"]
ano_inicial = 2010
ano_final = 2024
semana_inicial = 1
semana_final = 53

arquivo_csv = "dados_infodengue/infodengue_fortaleza_2010_2025.csv"
arquivo_relatorio = "dados_formatados/INFODENGUE_DADOS_FORTALEZA.CSV"

lista_dataframes = []

print("Iniciando a extração de dados da API do InfoDengue...")

for doenca in doencas:
    print(f"Coletando dados para: {doenca.upper()}...")

    # montando a url da API
    url = (
        "https://info.dengue.mat.br/api/alertcity?"
        f"geocode={geocode}"
        f"&disease={doenca}"
        f"&format=csv"
        f"&ew_start={semana_inicial}"
        f"&ew_end={semana_final}"
        f"&ey_start={ano_inicial}"
        f"&ey_end={ano_final}"
    )
    
    # Fazendo a requisição HTTP
    resposta = requests.get(url)
        
    # Verificando se a requisição foi bem sucedida (Status 200)
    if resposta.status_code == 200:
            # Lendo o CSV recebido diretamente em um DataFrame do Pandas
            dados_csv = io.StringIO(resposta.text)
            df_temp = pd.read_csv(dados_csv)
            
            # Adicionando uma coluna para identificar a doença neste lote de dados
            df_temp['tipo_doenca'] = doenca
            
            # Guardando o DataFrame na nossa lista
            lista_dataframes.append(df_temp)
    else:
            print(f"Erro ao coletar {doenca}. Status Code: {resposta.status_code}")
        
    # pequena pausa para não sobrecarregar a API pública
    time.sleep(2)

print("URL usada:")
print(url)

# Consolidação e Salvamento (Load)
if lista_dataframes:
    # Concatenando todos os DataFrames da lista em um único DataFrame
    df_consolidado = pd.concat(lista_dataframes, ignore_index=True)
    
    # Salvando o resultado final no seu arquivo de relatório
    df_consolidado.to_csv(arquivo_relatorio, index=False)
    print(f"\nExtração concluída! Dados salvos em: {arquivo_relatorio}")
    print(f"Total de registros coletados: {len(df_consolidado)} linhas.")
else:
    print("\nNenhum dado foi coletado. Verifique sua conexão ou os parâmetros da API.")

Iniciando a extração de dados da API do InfoDengue...
Coletando dados para: DENGUE...
Coletando dados para: ZIKA...
Coletando dados para: CHIKUNGUNYA...
URL usada:
https://info.dengue.mat.br/api/alertcity?geocode=2304400&disease=chikungunya&format=csv&ew_start=1&ew_end=53&ey_start=2010&ey_end=2024

Extração concluída! Dados salvos em: dados_formatados/INFODENGUE_DADOS_FORTALEZA.CSV
Total de registros coletados: 2301 linhas.


Início do processo de validação dos dados, selecionando as colunas relevantes para análise

In [2]:
#Pegando as colunas de interesse para o relatório
colunas_interesse = ["data_iniSE", "casos_est", "tipo_doenca", "nivel"]

# Gerando dataframe com as colunas de interesse
df_relatorio = df_consolidado[colunas_interesse]

#alterando o nome das colunas para o relatório
df_relatorio.rename(columns={"data_iniSE": "data", "casos_est": "casos_estimados"}, inplace=True)
    
print(df_relatorio.head(5))

         data  casos_estimados tipo_doenca  nivel
0  2024-12-22             42.0      dengue      1
1  2024-12-15             65.0      dengue      1
2  2024-12-08             82.0      dengue      1
3  2024-12-01             85.0      dengue      1
4  2024-11-24             99.0      dengue      1


Soma de caso obtidos por mês 

In [4]:
import pandas as pd

#Garantir que a coluna de data seja lida como "tempo" pelo Python
# (Substitua 'data' pelo nome real da sua coluna se for diferente, ex: 'data_iniSE')
df_relatorio['data'] = pd.to_datetime(df_relatorio['data'])

#Criar a Tabela Dinâmica (Pivot Table)
df_mensal = df_relatorio.pivot_table(
    index=pd.Grouper(key='data', freq='MS'), # Agrupa por Início do Mês (Month Start)
    columns='tipo_doenca',                   # Transforma as categorias em colunas
    values='casos_estimados',                # Os valores que serão somados
    aggfunc='sum',                           # A operação matemática (Soma)
    fill_value=0                             # Crucial: Preenche com 0 meses sem casos (ex: Zika em 2010)
).reset_index()

#Limpeza visual (remove o rótulo interno 'tipo_doenca' que fica acima das colunas)
df_mensal.columns.name = None

#Renomeando a coluna de data conforme você solicitou
df_mensal.rename(columns={'data': 'MES_REFERENCIA'}, inplace=True)

#Criando a coluna de Total (A nossa Variável Alvo)
# Confirme se os nomes das colunas geradas estão exatamente assim (minúsculas)
df_mensal['total_arboviroses'] = df_mensal['dengue'] + df_mensal['zika'] + df_mensal['chikungunya']

print(df_mensal.head(5))

  MES_REFERENCIA  chikungunya  dengue  zika  total_arboviroses
0     2010-01-01          0.0   388.0   0.0              388.0
1     2010-02-01          0.0   301.0   0.0              301.0
2     2010-03-01          0.0   337.0   0.0              337.0
3     2010-04-01          0.0   388.0   0.0              388.0
4     2010-05-01          0.0   514.0   0.0              514.0


Adicionando uma coluna de nivel de alerta

In [5]:
media_casos = df_mensal["total_arboviroses"].mean()
desvio_padrao = df_mensal["total_arboviroses"].std()

# Definição da função de alerta
def definir_alerta(casos):
    if casos <= media_casos:
        return 1 # Alerta baixo
    elif casos <= (media_casos + desvio_padrao):
        return 2 # Alerta médio
    elif casos <= (media_casos + 2 * desvio_padrao):
        return 3 # Alerta alto
    else:
        return 4 # Alerta crítico

# Criando a nova coluna de nível de alerta aplicando a função na coluna total
df_mensal["nivel_alerta"] = df_mensal['total_arboviroses'].apply(definir_alerta)

# Visualizando o resultado final estruturado
print(df_mensal.head())

  MES_REFERENCIA  chikungunya  dengue  zika  total_arboviroses  nivel_alerta
0     2010-01-01          0.0   388.0   0.0              388.0             1
1     2010-02-01          0.0   301.0   0.0              301.0             1
2     2010-03-01          0.0   337.0   0.0              337.0             1
3     2010-04-01          0.0   388.0   0.0              388.0             1
4     2010-05-01          0.0   514.0   0.0              514.0             1


Exporta o Dataframe

In [7]:
#exportando o dataframe para um arquivo csv
df_mensal.to_csv(arquivo_relatorio, index=False, encoding="utf-8")
print(f"\nRelatório salvo com sucesso: {arquivo_relatorio}")


Relatório salvo com sucesso: dados_formatados/INFODENGUE_DADOS_FORTALEZA.CSV
